In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
#checkpointer that saves the state in memory, this is useful for testing and debugging. for production use, we can use a checkpointer that saves the state in a database or a file, this one saves in RAM
#we have postgre , redis , mongodb checkpointer in langgraph, we can also create our own checkpointer by implementing the Checkpointer interface.

In [4]:
load_dotenv()

True

In [6]:
llm=ChatGroq(model="openai/gpt-oss-20b")

In [7]:

class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [8]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [9]:

def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [10]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

In [11]:
graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

In [12]:
checkpointer = InMemorySaver()

In [13]:
workflow = graph.compile(checkpointer=checkpointer)

In [17]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'fat person'}, config=config1)


{'topic': 'fat person',
 'joke': 'Why did the overweight person bring a ladder to the gym?  \nBecause they heard the “scale” was a step up in the workout!',
 'explanation': '**Explanation of the joke**\n\n> **Why did the overweight person bring a ladder to the gym?  \n>  Because they heard the “scale” was a step up in the workout!**\n\n---\n\n### 1. What’s going on in the set‑up?\n\n- The joke starts with a **silly, literal** image: an overweight person (someone who might be concerned with weight) bringing a **ladder** to a gym.  \n- The audience expects a conventional gym item—treadmill, dumbbells, etc.—but instead gets a ladder, which is obviously out of place.\n\n### 2. The punchline hinges on a **double meaning** (“wordplay”)\n\n- The key word is **“scale.”**  \n  1. **Weight scale** – the device you step on to check your body weight.  \n  2. **Scale** (in the sense of “step” or “ladder”) – a set of steps, or a ladder itself, used to climb or go “up.”\n\n- The overweight person *mi

In [18]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'fat person', 'joke': 'Why did the overweight person bring a ladder to the gym?  \nBecause they heard the “scale” was a step up in the workout!', 'explanation': '**Explanation of the joke**\n\n> **Why did the overweight person bring a ladder to the gym?  \n>  Because they heard the “scale” was a step up in the workout!**\n\n---\n\n### 1. What’s going on in the set‑up?\n\n- The joke starts with a **silly, literal** image: an overweight person (someone who might be concerned with weight) bringing a **ladder** to a gym.  \n- The audience expects a conventional gym item—treadmill, dumbbells, etc.—but instead gets a ladder, which is obviously out of place.\n\n### 2. The punchline hinges on a **double meaning** (“wordplay”)\n\n- The key word is **“scale.”**  \n  1. **Weight scale** – the device you step on to check your body weight.  \n  2. **Scale** (in the sense of “step” or “ladder”) – a set of steps, or a ladder itself, used to climb or go “up.”\n\n- The ov

In [19]:

list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'fat person', 'joke': 'Why did the overweight person bring a ladder to the gym?  \nBecause they heard the “scale” was a step up in the workout!', 'explanation': '**Explanation of the joke**\n\n> **Why did the overweight person bring a ladder to the gym?  \n>  Because they heard the “scale” was a step up in the workout!**\n\n---\n\n### 1. What’s going on in the set‑up?\n\n- The joke starts with a **silly, literal** image: an overweight person (someone who might be concerned with weight) bringing a **ladder** to a gym.  \n- The audience expects a conventional gym item—treadmill, dumbbells, etc.—but instead gets a ladder, which is obviously out of place.\n\n### 2. The punchline hinges on a **double meaning** (“wordplay”)\n\n- The key word is **“scale.”**  \n  1. **Weight scale** – the device you step on to check your body weight.  \n  2. **Scale** (in the sense of “step” or “ladder”) – a set of steps, or a ladder itself, used to climb or go “up.”\n\n- The o

In [20]:

config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti go to therapy?  \nBecause it had too many *knot* feelings!',
 'explanation': '**The punchline is a double‑pun that plays on two very different meanings of the word “knot.”**\n\n| Element | Literal meaning | Word‑play meaning |\n|---------|-----------------|-------------------|\n| **Spaghetti** | Long, thin strands of pasta that tend to tangle | A food that can literally form knots |\n| **Knot** | A tangle or twist in a strand of string or pasta | A homophone for “not,” the negation word |\n| **Feelings** | Emotions that people talk about in therapy | A humorous way to say “not feelings” – i.e., the pasta has no emotions |\n\nSo the joke unfolds like this:\n\n1. **Why did the spaghetti go to therapy?**  \n   - Therapy is a place people go to work through emotional problems. The absurd image of a pasta noodle seeking psychological help sets the stage.\n\n2. **Because it had too many *knot* feelings!**  \n   - **Knot** (the pasta’s tenden

In [21]:

workflow.get_state(config1)

StateSnapshot(values={'topic': 'fat person', 'joke': 'Why did the overweight person bring a ladder to the gym?  \nBecause they heard the “scale” was a step up in the workout!', 'explanation': '**Explanation of the joke**\n\n> **Why did the overweight person bring a ladder to the gym?  \n>  Because they heard the “scale” was a step up in the workout!**\n\n---\n\n### 1. What’s going on in the set‑up?\n\n- The joke starts with a **silly, literal** image: an overweight person (someone who might be concerned with weight) bringing a **ladder** to a gym.  \n- The audience expects a conventional gym item—treadmill, dumbbells, etc.—but instead gets a ladder, which is obviously out of place.\n\n### 2. The punchline hinges on a **double meaning** (“wordplay”)\n\n- The key word is **“scale.”**  \n  1. **Weight scale** – the device you step on to check your body weight.  \n  2. **Scale** (in the sense of “step” or “ladder”) – a set of steps, or a ladder itself, used to climb or go “up.”\n\n- The ov

In [22]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'fat person', 'joke': 'Why did the overweight person bring a ladder to the gym?  \nBecause they heard the “scale” was a step up in the workout!', 'explanation': '**Explanation of the joke**\n\n> **Why did the overweight person bring a ladder to the gym?  \n>  Because they heard the “scale” was a step up in the workout!**\n\n---\n\n### 1. What’s going on in the set‑up?\n\n- The joke starts with a **silly, literal** image: an overweight person (someone who might be concerned with weight) bringing a **ladder** to a gym.  \n- The audience expects a conventional gym item—treadmill, dumbbells, etc.—but instead gets a ladder, which is obviously out of place.\n\n### 2. The punchline hinges on a **double meaning** (“wordplay”)\n\n- The key word is **“scale.”**  \n  1. **Weight scale** – the device you step on to check your body weight.  \n  2. **Scale** (in the sense of “step” or “ladder”) – a set of steps, or a ladder itself, used to climb or go “up.”\n\n- The o